# A theoretical framework for compact and efficient quantum state simulation using Finite State Machines

This notebook provides a simple implementation of Mealy Quantum State Models and Mealy Quantum Gate Transducers, specifically targeted at the practical execution of the paper examples. It is written in pure Python so that no external libraries are required. The data structures used to represent both MQSM and MQGT are high-level Python native structures (dictionaries, lists, etc.), and they were designed having in mind:

- Accessibility to Python users (not necessarily experts in computer programming or high-performance software development), so that the code is easy to understand.
- Tight link to formal high-level descriptions of the paper (states, transitions, machines, etc.).

For these reasons, the code is not intended for use in High Performance Computing scenarios. 

According to the aforementioned criteria, I hope that the reader can use this notebook to fill in the gap between the formal and theoretical description given in the paper and how the structures work in practice. 

That said, I hope this code helps you to understand the core concepts behind MQSMs and MQGTs. It was built with that sole purpose.


In [12]:
import math # The only required library to run the examples

## Mealy Quantum State Model implementation

It is provided in the class MQSM. Self-contained with explanatory comments.

In [13]:

"""
Main class to represent a Mealy Quantum State Model
"""
class MQSM:
    """
    Internal representation:
        self.nqb: Number of qubits in the state (not necessary, but helpful)
        self.states: List of states in the machine (type MQSMState)
        self.root: Points to the initial state (not necessary, but helpful)
        self.s_t: Points to the final state (not necessary, but helpful)
        self._state_counter: Number of states (not necessary, but helpful to build the state state_ids)
    """
    
    
    """
    subclass to model a machine state of a Mealy Quantum State Model
    """
    class MQSMState:
        """
        Internal representation:
            self.transitions: Dictionary (only two elements 0/1 are used)
                self.transitions[0]= weight of transition for qubit state |0>
                self.transitions[1]= weight of transition for qubit state |1>
        """
        
        
        """
        Constructor. Builds a state
            state_id: A state/state identifier (for plotting purposes)
            depth: Depth of the state in the MQSM graph representation
        """
        def __init__(self, state_id, depth):
            self.id = state_id
            self.depth = depth
            self.transitions = {}
            

        """
        Appends a transition delta(self, bit)=next_state with omega(self, bit=weight)
        """
        def add_transition(self, bit, weight, next_state):
            self.transitions[bit] = (weight, next_state)

    
    
    
    """
    Constructor. Builds a basis state of nqb qubits |basis>_{nqb}
    """
    def __init__(self, basis=None, nqb=None):
        self.nqb = nqb
        self.states = []
        self.root = None
        self.s_T = None
        self._state_counter = 0

        # Si se proporcionan argumentos, construye un estado base
        if basis is not None and nqb is not None:
            self._build_basis(basis, nqb)



    """
    Helper function to build a new machine state at depth depth, and stating if it is terminal or not
    """
    def _get_new_state(self, depth, is_terminal=False):
        
        # Create the state label (state_id)
        state_id = "T" if is_terminal else str(self._state_counter)
        if not is_terminal:
            self._state_counter += 1
            
        # Builds the state and appends it to the machine state list
        state = MQSM.MQSMState(state_id, depth)
        self.states.append(state)
        return state


    
    """
    Builds the basis state |basis>_{nqb} of nqb qubits on self
    """
    def _build_basis(self, basis, nqb):
        # Get the final state
        self.s_T = self._get_new_state(nqb, is_terminal=True)
        
        # Get the state of the first qubit (s_0)
        self.root = self._get_new_state(0)
        
        current = self.root
        for i in range(nqb):
            # Calculate if current qubit is |0> or |1>
            bit = (basis >> (nqb - 1 - i)) & 1
            
            # Insert transition to next qubit or s_t if the current qubit is the last one
            next_state = self.s_T if i == nqb - 1 else self._get_new_state(i + 1)
            current.add_transition(bit, 1.0, next_state)
            
            # Go to next qubit
            current = next_state



    """
    Creates a MQSM separable state with len(label) qubits. Each qubit can be:
        '0' -> |0>
        '1' -> |1>
        '+' -> |+>
        '-' -> |->
        'r' -> |i>
        'l' -> |-i>
    """
    @staticmethod
    def from_label(label):
        
        # get the number of qubits and build the MQSM skeleton
        nqb = len(label)
        mqsm = MQSM(nqb=nqb)
        
        # Builds the terminal state
        mqsm.s_T = mqsm._get_new_state(nqb, is_terminal=True)
        
        # Builds the initial state (s_0) for qubit q0
        mqsm.root = mqsm._get_new_state(0)
        
        inv_sqrt2 = 1 / math.sqrt(2) # To avoid calling to 1/math.sqrt(2) multiple times
        
        # Iterate for each qubit
        current = mqsm.root
        for i, char in enumerate(label):
            
            # The next state of the last qubit is s_T. Otherwhise, is a new state for the next qubit
            next_state = mqsm.s_T if i == nqb - 1 else mqsm._get_new_state(i + 1)
            
            # Create the corresponding transitions.
            if char == '0':
                current.add_transition(0, 1.0, next_state)
            elif char == '1':
                current.add_transition(1, 1.0, next_state)
            elif char == '+':
                current.add_transition(0, inv_sqrt2, next_state)
                current.add_transition(1, inv_sqrt2, next_state)
            elif char == '-':
                current.add_transition(0, inv_sqrt2, next_state)
                current.add_transition(1, -inv_sqrt2, next_state)
            elif char == 'r':
                current.add_transition(0, inv_sqrt2, next_state)
                current.add_transition(1, inv_sqrt2 * 1j, next_state)
            elif char == 'l':
                current.add_transition(0, inv_sqrt2, next_state)
                current.add_transition(1, -inv_sqrt2 * 1j, next_state)
            else:
                raise ValueError("Error: {} is not a valid state for from_label".format(char))
                
            current = next_state # Move to next qubit
        return mqsm


    """
    Print the MQSM in statevector format. 
    """
    def print(self):
        
        statevector = [] # sv representation
        
        """
        Helper sub-function to create the statevector recursively
        """
        def dfs(state, current_ket, current_amp):
            if state.id == "T":
                if not math.isclose(abs(current_amp), 0.0, abs_tol=1e-9):
                    statevector.append((current_ket, current_amp))
                return
            
            for bit, (weight, next_state) in state.transitions.items():
                dfs(next_state, current_ket + str(bit), current_amp * weight)

        dfs(self.root, "", 1.0)
        
        # Visual formatting
        three_decimals= "{:.3f}"
        terms = []
        for ket, amp in statevector:
            # Limpiar el formato del número complejo/real
            if isinstance(amp, complex):
                if math.isclose(amp.imag, 0.0, abs_tol=1e-9):
                    amp_str = three_decimals.format(amp.real)
                elif math.isclose(amp.real, 0.0, abs_tol=1e-9):
                    amp_str = three_decimals.format(amp.imag)+"i"
                else:
                    amp_str = three_decimals.format(amp)
            else:
                amp_str = three_decimals.format(amp)
                
            terms.append("({})|{}>".format(amp_str, ket))
            
        # Print the statevector
        print(" + ".join(terms) if terms else "0")





## Mealy Quantum Gate Transducer implementation

It is provided in the class MQGT. Self-contained with explanatory comments.

**Author's note**: The evolve method is not complete and it is highly inefficient. It was built to show how the paper examples work in practice.

In [14]:
"""
Main class to represent a Mealy Quantum Gate Transducer
"""
class MQGT:
    """
    Internal representation:
        self.num_qubits: The number of qubits involved in the machine (not necessary, but helpful)
        self.states: List of machine states
        self.u_0: The initial state
        self.u_T: The final state
        
    """
    
    
    """
    Subclass to model a machine state of a Mealy Quantum Gate Transducer
    """
    class MQGTState:
        """ 
        Internal representation:
            self.id, a node identifier (mainly for plotting, but helpful anyway)
            self.transitions (similar to MQSM)
        """
        
        """
        Constructor: builds a MQGT state with its identifier and empty transitions
        """
        def __init__(self, state_id):
            self.id = state_id
            self.transitions = {}  # Dict: (y, x) -> (weight, next_state)

        """
        Appends transition delta(self, (y,x))= next_state with omega(self, (y,x)= weight)
        """
        def add_transition(self, y, x, weight, next_state):
            self.transitions[(y, x)] = (weight, next_state)

    
    
    """
    Constructor. Builds an empty machine to process num_qubits qubits
    """
    def __init__(self, num_qubits):
        self.num_qubits = num_qubits
        self.states = []
        self.u_0 = None
        self.u_T = MQGT.MQGTState("u_T")
        self.states.append(self.u_T)
        self._state_counter = 0


    """
    Helper function to create a new machine state. Similar to MQSMs
    """
    def _get_new_state(self):
        state= MQGT.MQGTState("u_{}".format(self._state_counter))
        self._state_counter += 1
        self.states.append(state)
        return state


    """ ############################################################################################
    Gate Library. It only contains the following gates just for illustrative purposes:
        H() -> gets the Hadamard MQGT (1-qb)
        X() -> gets the X MQGT (1-qb)
        CNOT() -> gets the Controlled-X MQGT (2-qb)
        CZ() -> gets the Controlled-Z MQGT (2-qb)
        CH() -> gets the Controlled-H MQGT (2-qb)
        SWAP() -> gets the SWAP MQFT (2-qb)
    ############################################################################################# """
    
    """
    Hadamard MQGT
    """
    @staticmethod
    def H():
        gate = MQGT(1)
        u0 = gate._get_new_state()
        gate.u_0 = u0
        inv_sqrt2 = 1 / math.sqrt(2)
        
        u0.add_transition(0, 0, inv_sqrt2, gate.u_T)
        u0.add_transition(0, 1, inv_sqrt2, gate.u_T)
        u0.add_transition(1, 0, inv_sqrt2, gate.u_T)
        u0.add_transition(1, 1, -inv_sqrt2, gate.u_T)
        return gate


    """
    Pauli-X MQGT
    """
    @staticmethod
    def X():
        gate = MQGT(1)
        u0 = gate._get_new_state()
        gate.u_0 = u0
        
        u0.add_transition(0, 1, 1, gate.u_T)
        u0.add_transition(1, 0, 1, gate.u_T)
        return gate


    """ 
    Controlled-X MQGT
    """
    @staticmethod
    def CNOT():
        gate = MQGT(2)
        u0 = gate._get_new_state()
        gate.u_0 = u0
        u_Id = gate._get_new_state()
        u_X = gate._get_new_state()
        
        u0.add_transition(0, 0, 1.0, u_Id)
        u0.add_transition(1, 1, 1.0, u_X)
        
        u_Id.add_transition(0, 0, 1.0, gate.u_T)
        u_Id.add_transition(1, 1, 1.0, gate.u_T)
        
        u_X.add_transition(0, 1, 1.0, gate.u_T)
        u_X.add_transition(1, 0, 1.0, gate.u_T)
        return gate

    """ 
    Controlled-H MQGT
    """
    @staticmethod
    def CH():
        gate = MQGT(2)
        u0 = gate._get_new_state()
        gate.u_0 = u0
        u_Id = gate._get_new_state()
        u_H = gate._get_new_state()
        
        u0.add_transition(0, 0, 1.0, u_Id)
        u0.add_transition(1, 1, 1.0, u_H)
        
        u_Id.add_transition(0, 0, 1.0, gate.u_T)
        u_Id.add_transition(1, 1, 1.0, gate.u_T)
        
        inv_sqrt2 = 1 / math.sqrt(2)
        
        u_H.add_transition(0, 0, inv_sqrt2, gate.u_T)
        u_H.add_transition(0, 1, inv_sqrt2, gate.u_T)
        u_H.add_transition(1, 0, inv_sqrt2, gate.u_T)
        u_H.add_transition(1, 1, -inv_sqrt2, gate.u_T)
        return gate


    """ 
    Controlled-Z MQGT
    """
    @staticmethod
    def CZ():
        gate = MQGT(2)
        u0 = gate._get_new_state()
        gate.u_0 = u0
        u_Id = gate._get_new_state()
        u_Z = gate._get_new_state()
        
        u0.add_transition(0, 0, 1.0, u_Id)
        u0.add_transition(1, 1, 1.0, u_Z)
        
        u_Id.add_transition(0, 0, 1.0, gate.u_T)
        u_Id.add_transition(1, 1, 1.0, gate.u_T)
        
        u_Z.add_transition(0, 0, 1.0, gate.u_T)
        u_Z.add_transition(1, 1, -1.0, gate.u_T)
        return gate


    """ 
    Controlled-Controlled-Z MQGT
    """
    @staticmethod
    def CCZ():
        gate = MQGT(3)
        u0 = gate._get_new_state()
        gate.u_0 = u0
        u_Id1 = gate._get_new_state()
        u_Id2 = gate._get_new_state()
        u_c2= gate._get_new_state()
        u_Z = gate._get_new_state()
        
        u0.add_transition(0, 0, 1.0, u_Id1)
        u0.add_transition(1, 1, 1.0, u_c2)
        
        u_Id1.add_transition(0, 0, 1.0, u_Id2)
        u_Id1.add_transition(1, 1, 1.0, u_Id2)

        u_c2.add_transition(0, 0, 1.0, u_Id2)
        u_c2.add_transition(1, 1, 1.0, u_Z)

        u_Id2.add_transition(0, 0, 1.0, gate.u_T)
        u_Id2.add_transition(1, 1, 1.0, gate.u_T)
        
        u_Z.add_transition(0, 0, 1.0, gate.u_T)
        u_Z.add_transition(1, 1, -1.0, gate.u_T)
        return gate


    """
    SWAP MQGT
    """
    @staticmethod
    def SWAP():
        """Compuerta SWAP (2 qubits)."""
        gate = MQGT(2)
        u0 = gate._get_new_state()
        gate.u_0 = u0
        u1 = gate._get_new_state()
        u2 = gate._get_new_state()
        u3 = gate._get_new_state()
        u4 = gate._get_new_state()
        
        u0.add_transition(0, 0, 1.0, u1)
        u0.add_transition(1, 0, 1.0, u2)
        u0.add_transition(0, 1, 1.0, u3)
        u0.add_transition(1, 1, 1.0, u4)
        
        u1.add_transition(0, 0, 1.0, gate.u_T)
        u2.add_transition(0, 1, 1.0, gate.u_T)
        u3.add_transition(1, 0, 1.0, gate.u_T)
        u4.add_transition(1, 1, 1.0, gate.u_T)
        return gate



    """
    Helper function to print the Unitary Matrix of the MQGT (for debugging)
    """
    def print(self):
        
        # Get matrix size and representation as list of lists
        size = 1 << self.num_qubits
        matrix = [[0.0] * size for _ in range(size)]
        
        
        """
        Helper function to traverse the graph and get the matrix entries
        """
        def traverse(u, depth, current_y, current_x, current_w):
            if u.id == "u_T" or depth == self.num_qubits:
                matrix[current_y][current_x] += current_w
                return
                
            for (y, x), (w, next_u) in u.transitions.items():
                traverse(next_u, depth + 1, (current_y << 1) | y, (current_x << 1) | x, current_w * w)
                
        # Get the matrix entries
        traverse(self.u_0, 0, 0, 0, 1.0)
        
        
        # Print the matrix
        for row in matrix:
            formatted_row = []
            for val in row: # process a row, using fixed length for each entry
                if isinstance(val, complex):
                    formatted_row.append("{:5.2f}+{:5.2f}j".format(val.real, val.imag))
                else:
                    formatted_row.append("{:5.2f}".format(val))
            print("  ".join(formatted_row)) # Print a row




    """
    Evolves an input MQSM with self, acting on qubits "qubits"
    """
    def evolve(self, mqsm, qubits):

        """ 
        I feel lazy. Current implementation of evolve does not allow reverse controlled gates
        The next lines make a SWAP so that the qubit ordering is [control, target]
        """
        if list(qubits) != sorted(qubits):
            target_qubits = sorted(qubits)
            working_qubits = list(qubits)
            swaps_to_do = []
            
            # 1. Calcular las permutaciones necesarias
            for i in range(len(working_qubits)):
                if working_qubits[i] != target_qubits[i]:
                    idx = working_qubits.index(target_qubits[i])
                    q_a = working_qubits[i]
                    q_b = working_qubits[idx]
                    swaps_to_do.append((min(q_a, q_b), max(q_a, q_b)))
                    working_qubits[i], working_qubits[idx] = working_qubits[idx], working_qubits[i]
                    
            swap_gate = MQGT.SWAP()
            current_mqsm = mqsm
            
            for q_a, q_b in swaps_to_do: # Do the swap
                current_mqsm = swap_gate.evolve(current_mqsm, [q_a, q_b])
                
            # Do the state evolution
            current_mqsm = self.evolve(current_mqsm, target_qubits)
            
            
            for q_a, q_b in reversed(swaps_to_do): # Reverse swap
                current_mqsm = swap_gate.evolve(current_mqsm, [q_a, q_b])
                
            return current_mqsm
        
        # Do the evolution assuming qubits are given in order
        qubits = sorted(qubits)
        new_mqsm = MQSM(nqb=mqsm.nqb) 
        new_mqsm.s_T = new_mqsm._get_new_state(mqsm.nqb, is_terminal=True)
        
        # Cache to store compression isomorphism information
        cache = {d: {} for d in range(mqsm.nqb + 1)}
        compose_cache = {}


        """
        Helper to format complex numbers to avoid rounding errors
        """
        def format_complex(c):
            if isinstance(c, complex): return (round(c.real, 6), round(c.imag, 6))
            return (round(c, 6), 0.0)


        """
        Do the machine  normalization and compression
        """
        def normalize_and_get_state(w0, s0, w1, s1, depth):
            if abs(w0) < 1e-9 and abs(w1) < 1e-9:
                return 0.0, None
                
            norm = math.sqrt(abs(w0)**2 + abs(w1)**2)
            phase = (w0 / abs(w0)) if abs(w0) > 1e-9 else (w1 / abs(w1))
            
            C = norm * phase
            wnew0 = w0 / C if abs(w0) > 1e-9 else 0.0
            wnew1 = w1 / C if abs(w1) > 1e-9 else 0.0
            
            trans_key = []
            if abs(wnew0) > 1e-9 and s0 is not None:
                trans_key.append((0, format_complex(wnew0), s0.id))
            if abs(wnew1) > 1e-9 and s1 is not None:
                trans_key.append((1, format_complex(wnew1), s1.id))
            trans_key = tuple(trans_key)
            
            if trans_key in cache[depth]:
                return C, cache[depth][trans_key]
                
            s_R = new_mqsm._get_new_state(depth)
            if abs(wnew0) > 1e-9 and s0 is not None: s_R.add_transition(0, wnew0, s0)
            if abs(wnew1) > 1e-9 and s1 is not None: s_R.add_transition(1, wnew1, s1)
                
            cache[depth][trans_key] = s_R
            return C, s_R


        """
        Machine Addition operator. Highly inefficient (recursive)
        """
        def add_machines(cA, sA, cB, sB, depth):
            if abs(cA) < 1e-9: return cB, sB
            if abs(cB) < 1e-9: return cA, sA
            if sA == sB: return cA + cB, sA
            
            w0A, n0A = sA.transitions.get(0, (0.0, None))
            w0B, n0B = sB.transitions.get(0, (0.0, None))
            w0, s0 = add_machines(cA * w0A, n0A, cB * w0B, n0B, depth + 1)
            
            w1A, n1A = sA.transitions.get(1, (0.0, None))
            w1B, n1B = sB.transitions.get(1, (0.0, None))
            w1, s1 = add_machines(cA * w1A, n1A, cB * w1B, n1B, depth + 1)
            
            return normalize_and_get_state(w0, s0, w1, s1, depth)


        """
        gate-state composition. It generates the new paths (y,x) through cross product u * s
        """
        def compose(u, s, depth):
            if depth == mqsm.nqb:
                return 1.0, new_mqsm.s_T
                
            cache_key = (u.id if u else None, s.id, depth)
            if cache_key in compose_cache:
                return compose_cache[cache_key]
                
            is_gate_active = depth in qubits
            branches_for_y = {0: [], 1: []}
            
            for y in (0, 1):
                for x in (0, 1):
                    if x not in s.transitions: continue
                    ws, s_next = s.transitions[x]
                    
                    if is_gate_active:
                        if (y, x) not in u.transitions: continue
                        wu, u_next = u.transitions[(y, x)]
                    else:
                        if y != x: continue  # Salto por Identidad implícita
                        wu, u_next = 1.0, u
                        
                    c_branch = ws * wu
                    if abs(c_branch) < 1e-9: continue
                        
                    c_sub, s_sub = compose(u_next, s_next, depth + 1)
                    branches_for_y[y].append((c_branch * c_sub, s_sub))
            
            out_branches = {}
            for y in (0, 1):
                br_list = branches_for_y[y]
                if not br_list:
                    out_branches[y] = (0.0, None)
                elif len(br_list) == 1:
                    out_branches[y] = br_list[0]
                else:
                    w_curr, s_curr = add_machines(br_list[0][0], br_list[0][1], br_list[1][0], br_list[1][1], depth + 1)
                    for b in br_list[2:]:
                        w_curr, s_curr = add_machines(w_curr, s_curr, b[0], b[1], depth + 1)
                    out_branches[y] = (w_curr, s_curr)
                    
            w0, s0 = out_branches[0]
            w1, s1 = out_branches[1]
            
            C, s_R = normalize_and_get_state(w0, s0, w1, s1, depth)
            compose_cache[cache_key] = (C, s_R)
            return C, s_R


        # Do the state composition
        C_final, root_final = compose(self.u_0, mqsm.root, 0)
        
        # Do final rounding and normalization
        if abs(C_final - 1.0) > 1e-9 and root_final is not None:
            actual_root = new_mqsm._get_new_state(0)
            for bit, (w, nxt) in root_final.transitions.items():
                actual_root.add_transition(bit, w * C_final, nxt)
            new_mqsm.root = actual_root
        else:
            new_mqsm.root = root_final

        return new_mqsm


# Examples

Paper examples to evolve MQSMs with MQGTs.

## Example: H gate

In [15]:
# The H gate MQGT is defined above in the class MQGT according to the paper's diagram
h= MQGT.H()
print('\nHadamard gate unitary matrix:')
h.print()


Hadamard gate unitary matrix:
 0.71   0.71
 0.71  -0.71


## Example: CNOT gate

In [16]:
# The CNOT gate MQGT is defined above in the class MQGT according to the paper's diagram
cnot= MQGT.CNOT()
print('\nCNOT gate unitary matrix:')
cnot.print()


CNOT gate unitary matrix:
 1.00   0.00   0.00   0.00
 0.00   1.00   0.00   0.00
 0.00   0.00   0.00   1.00
 0.00   0.00   1.00   0.00


## Example: Evolution of |+10> with CNOT(0,2)



In [17]:
# State preparation
ket= MQSM.from_label("+10")
print('Initial state:')
ket.print()

# Apply CNOT with control on qubit 0 and target on qubit 2
cnot = MQGT.CNOT()
new_ket = cnot.evolve(ket, [0, 2])

print('Final state:')
new_ket.print()

Initial state:
(0.707)|010> + (0.707)|110>
Final state:
(0.707)|010> + (0.707)|111>


## Example: Evolution of (|01+>+|100>)/sqrt{2} with SWAP(0,2)

In [18]:
# State preparation
ket= MQSM.from_label("+1+")
cnot= MQGT.CNOT()
ch= MQGT.CH()
new_ket = cnot.evolve(ket, [0, 1])
new_ket = ch.evolve(new_ket, [0, 2])

print('Initial state:')
new_ket.print()

# Do the swap
swap= MQGT.SWAP()
new_ket = swap.evolve(new_ket, [0, 2])

print('Final state:')
new_ket.print()

Initial state:
(0.500)|010> + (0.500)|011> + (0.707)|100>
Final state:
(0.707)|001> + (0.500)|010> + (0.500)|110>


# Example: Phase kickback

In [19]:
ket= MQSM.from_label("0-")
print('Initial state:')
ket.print()

h= MQGT.H()
new_ket = h.evolve(ket, [0])

cnot= MQGT.CNOT()
new_ket = cnot.evolve(new_ket, [0, 1])

new_ket= h.evolve(new_ket, [0])

print('After phase kickback:')
new_ket.print()

Initial state:
(0.707)|00> + (-0.707)|01>
After phase kickback:
(0.707)|10> + (-0.707)|11>


## Example: Grover search

In [20]:
ket= MQSM.from_label("00") # Two qubits + ancilla

print('Initial state:')
ket.print()

# Standaard superposition + Oracle marks |11>
h= MQGT.H()
x= MQGT.X()
cz= MQGT.CZ()
new_ket = h.evolve(ket, [0])
new_ket = h.evolve(new_ket, [1])
new_ket = cz.evolve(new_ket, [0, 1])

print('After oracle:')
new_ket.print()


# Grover Diffuser
new_ket = h.evolve(new_ket, [0])
new_ket = h.evolve(new_ket, [1])
new_ket = x.evolve(new_ket, [0])
new_ket = x.evolve(new_ket, [1])
new_ket= cz.evolve(new_ket, [0, 1])
new_ket = x.evolve(new_ket, [0])
new_ket = x.evolve(new_ket, [1])
new_ket = h.evolve(new_ket, [0])
new_ket = h.evolve(new_ket, [1])

# Oracle uncomputation
new_ket = cz.evolve(new_ket, [0, 1])

print('Final state:')
new_ket.print()

Initial state:
(1.000)|00>
After oracle:
(0.500)|00> + (0.500)|01> + (0.500)|10> + (-0.500)|11>
Final state:
(1.000)|11>
